## 1. Configuración e Importaciones
En esta celda importamos las librerías y definimos las constantes del proyecto (nombres de datasets, algoritmos, etc.).

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from scipy import stats

# --- CONFIGURACIÓN DE RUTAS ---
# Ajusta estas rutas según tu estructura de carpetas
PATH_DATOS = "./datos/"          # Donde están los csv de test (ej: iris_original_test_fold1.csv)
PATH_MODELOS = "./modelos/"      # Donde están los .pkl (ej: model_knn_original_fold1.pkl)
PATH_PREDICCIONES = "./predicciones/" # Donde se guardarán los resultados crudos

# Crear carpeta de salida si no existe
if not os.path.exists(PATH_PREDICCIONES):
    os.makedirs(PATH_PREDICCIONES)

# --- LISTAS DE CONTROL ---
DATASETS = [
    "original", "estandarizado", "normalizado",
    "originalPCA95", "originalPCA80",
    "estandarizadoPCA95", "estandarizadoPCA80",
    "normalizadoPCA95", "normalizadoPCA80"
]

# Algoritmos base solicitados
ALGORITMOS = ["knn", "svm", "naive_bayes", "random_forest"]

# 5 Iteraciones de validación cruzada
FOLDS = [1, 2, 3, 4, 5] 

print("Configuración cargada. Las predicciones se guardarán en:", PATH_PREDICCIONES)

## 2. Funciones Auxiliares (Carga y Guardado)
Necesitamos funciones para cargar los datos y modelos correspondientes a cada iteración y almacenar los resultados.

In [ ]:
def cargar_datos_test(dataset, fold):
    """
    Carga el conjunto de test para una iteración específica.
    Formato esperado del archivo: iris_{dataset}_test_fold{fold}.csv
    """
    filename = f"{PATH_DATOS}iris_{dataset}_test_fold{fold}.csv"
    try:
        df = pd.read_csv(filename)
        X_test = df.iloc[:, :-1].values # Todas las columnas menos la última (features)
        y_test = df.iloc[:, -1].values  # Última columna (target)
        return X_test, y_test
    except FileNotFoundError:
        print(f"⚠ ADVERTENCIA: No se encontró el archivo de datos: {filename}")
        return None, None

def cargar_modelo(dataset, algo, fold):
    """
    Carga el modelo entrenado (.pkl).
    Formato esperado: model_{algo}_{dataset}_fold{fold}.pkl
    """
    filename = f"{PATH_MODELOS}model_{algo}_{dataset}_fold{fold}.pkl"
    try:
        return joblib.load(filename)
    except FileNotFoundError:
        print(f"⚠ ADVERTENCIA: No se encontró el modelo: {filename}")
        return None

def guardar_prediccion(dataset, fold, metodo, y_true, y_pred, y_proba):
    """
    Guarda las predicciones y probabilidades en un CSV para que results.ipynb las procese.
    [cite_start]Formato de salida: pred_{fold}_{dataset}_{metodo}.csv [cite: 48]
    """
    # Estructura base con etiquetas reales y predicciones de clase
    data = {
        "y_true": y_true,
        "y_pred": y_pred
    }
    
    # Añadir probabilidades de pertenencia a cada clase (prob_0, prob_1...)
    # [cite_start]Necesario para calcular curva ROC y AUC después [cite: 52, 58]
    if y_proba is not None:
        # Asumimos que y_proba es una matriz (n_muestras, n_clases)
        for i in range(y_proba.shape[1]):
            data[f"prob_{i}"] = y_proba[:, i]
    
    df_out = pd.DataFrame(data)
    
    # Nombre del archivo parametrizado
    filename = f"{PATH_PREDICCIONES}pred_{fold}_{dataset}_{metodo}.csv"
    df_out.to_csv(filename, index=False)

## 3. Bucle Principal y Ensembles
Esta función implementa todas las fórmulas de la tabla del PDF. Al ser Iris un problema multiclase (3 clases), calculamos las métricas para cada clase y hacemos la media (Macro-average), que es el estándar cuando se piden estas fórmulas binarias en problemas multi-clase.

In [ ]:
print("--- INICIANDO GENERACIÓN DE PREDICCIONES Y ENSEMBLES ---")

for dataset in DATASETS:
    print(f"Procesando Dataset: {dataset}...")
    
    for fold in FOLDS:
        # 1. Cargar datos de test de este fold
        X_test, y_test = cargar_datos_test(dataset, fold)
        
        if X_test is None:
            continue
        
        # Listas para almacenar las predicciones de los modelos base (para luego hacer ensembles)
        ensemble_clases = [] # Para Votación
        ensemble_probas = [] # Para Media/Mediana
        modelos_exitosos = 0
        
        # --- FASE A: MODELOS INDIVIDUALES ---
        for algo in ALGORITMOS:
            model = cargar_modelo(dataset, algo, fold)
            
            if model:
                try:
                    # Predicción de clase (0, 1, 2)
                    y_pred = model.predict(X_test)
                    
                    # [cite_start]Predicción de probabilidad (necesario para AUC y Ensembles Media/Mediana) [cite: 58]
                    # NOTA: Asegúrate de que SVM se entrenó con probability=True
                    if hasattr(model, "predict_proba"):
                        y_proba = model.predict_proba(X_test)
                    else:
                        # Si el modelo no soporta proba, rellenamos con ceros (fallback)
                        y_proba = np.zeros((len(y_pred), len(np.unique(y_test))))
                    
                    # Guardar archivo individual
                    guardar_prediccion(dataset, fold, algo, y_test, y_pred, y_proba)
                    
                    # Guardar en memoria para calcular ensembles
                    ensemble_clases.append(y_pred)
                    ensemble_probas.append(y_proba)
                    modelos_exitosos += 1
                    
                except Exception as e:
                    print(f"Error ejecutando {algo} en {dataset} fold {fold}: {e}")
        
        # --- FASE B: ENSEMBLES ---
        # Solo ejecutamos ensembles si tenemos los 4 modelos base (KNN, SVM, NB, RF)
        if modelos_exitosos == 4:
            
            # [cite_start]1. ENSEMBLE VOTACIÓN [cite: 73]
            # "La predicción del ensemble será la clase predicha que tenga más votos"
            matriz_votos = np.array(ensemble_clases) # Shape: (4_modelos, n_muestras)
            # Calculamos la moda por columnas (axis=0)
            moda_result = stats.mode(matriz_votos, axis=0, keepdims=True)
            y_pred_votacion = moda_result[0][0]
            
            # Para guardar un archivo consistente, usamos el promedio de probas como "probabilidad" del ensemble
            y_proba_votacion = np.mean(np.array(ensemble_probas), axis=0)
            
            guardar_prediccion(dataset, fold, "ensemble_votacion", y_test, y_pred_votacion, y_proba_votacion)
            
            # [cite_start]2. ENSEMBLE MEDIA [cite: 75]
            # "La predicción del ensemble será la media de dichas probabilidades"
            matriz_probas = np.array(ensemble_probas) # Shape: (4_modelos, n_muestras, n_clases)
            y_proba_media = np.mean(matriz_probas, axis=0)
            # La clase final es la que tiene la probabilidad media más alta
            y_pred_media = np.argmax(y_proba_media, axis=1)
            
            guardar_prediccion(dataset, fold, "ensemble_media", y_test, y_pred_media, y_proba_media)
            
            # [cite_start]3. ENSEMBLE MEDIANA [cite: 78]
            # "Igual que el ensemble Media, pero usando la función mediana"
            y_proba_mediana = np.median(matriz_probas, axis=0)
            y_pred_mediana = np.argmax(y_proba_mediana, axis=1)
            
            guardar_prediccion(dataset, fold, "ensemble_mediana", y_test, y_pred_mediana, y_proba_mediana)
            
        else:
            print(f"Saltando Ensembles para {dataset} fold {fold}: Falta alguno de los 4 modelos base.")

print("--- PROCESO COMPLETADO ---")
print(f"Archivos de predicción guardados en: {PATH_PREDICCIONES}")